# Notebook 2: Inspect, Iterate, Train

**Phases 3–4** | ~45 minutes | **The heart of the workshop**

In this notebook you will:
1. Load your auto-labeled dataset into FiftyOne for deep inspection
2. Run a student model and evaluate it against the teacher's labels
3. Analyze error distributions — find WHERE the model fails
4. Use brain methods to find likely label errors and redundant images
5. Visualize dataset structure with embeddings
6. Curate the dataset (filter tiny labels) and compare before/after
7. Train a YOLO26n student model on curated labels
8. Evaluate the trained model and reflect

> **The data-centric insight**: The model is fine. The DATA is what you iterate on.
> Instead of tuning architectures or hyperparameters, we will find and fix specific
> data problems: wrong labels, redundant images, size-dependent failures.

> **AI assistance**: Use your preferred AI coding tool for guidance.
> See `cv_copilot_skill.md` for a ready-to-use system prompt.

---
## Section 0: Setup & Load Dataset into FiftyOne

In [ ]:
# ── Cell 0a: Environment + Path Setup ─────────────────────────────────────────────────
import sys
import glob
import warnings
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

# ── Device ───────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Python:  {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"Device:  {DEVICE}")

# ── Paths ────────────────────────────────────────────────────────────────────
WORKSHOP_DIR = Path(".").resolve().parent
DATA_DIR = WORKSHOP_DIR / "data"

# ============================================================
# CONFIGURE: Point this to YOUR labeled dataset from Notebook 1
# ============================================================
DATASET_DIR = DATA_DIR / "ppe_dataset_sam3_exp_a"  # <-- adjust if you used a different name
CLASS_NAMES = {0: "hardhat", 1: "person"}
DATASET_NAME = "ppe-nb02-analysis"

# Find trained weights (from a previous training run, if any)
runs_dir = DATA_DIR / "ppe_results"
run_dirs = sorted(glob.glob(str(runs_dir / "*/weights/best.pt")))
BEST_WEIGHTS = Path(run_dirs[-1]) if run_dirs else None

print(f"\nWorkshop dir:  {WORKSHOP_DIR}")
print(f"Dataset dir:   {DATASET_DIR}")
print(f"Best weights:  {BEST_WEIGHTS}")
print(f"Classes:       {CLASS_NAMES}")

assert DATASET_DIR.exists(), f"Dataset not found at {DATASET_DIR} -- run Notebook 1 first!"


In [ ]:
# ── Cell 0b: Utility — plot_gt_vs_pred() ──────────────────────────────────────
# This helper visualizes ground truth vs predictions side by side.
# You will use it throughout this notebook — run it once now.

from PIL import Image

def plot_gt_vs_pred(sample, class_colors=None, figsize=(16, 6)):
    """Plot ground truth and predictions side by side for one FiftyOne sample."""
    if class_colors is None:
        class_colors = {
            "hardhat": "#27ae60", "person": "#3498db",
            "bottle": "#27ae60", "cap": "#3498db",
        }

    img = np.array(Image.open(sample.filepath))
    h, w = img.shape[:2]

    fig, (ax_gt, ax_pred) = plt.subplots(1, 2, figsize=figsize)

    for ax, field, title in [
        (ax_gt, "ground_truth", "Ground Truth (teacher labels)"),
        (ax_pred, "predictions", "Predictions (student model)"),
    ]:
        ax.imshow(img)
        detections = sample[field]
        if detections and detections.detections:
            for det in detections.detections:
                x, y, bw, bh = det.bounding_box  # FiftyOne: relative [x, y, w, h]
                x1, y1 = int(x * w), int(y * h)
                x2, y2 = int((x + bw) * w), int((y + bh) * h)
                color = class_colors.get(det.label, "#e74c3c")
                rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                     linewidth=2, edgecolor=color, facecolor="none")
                ax.add_patch(rect)
                conf_str = f" {det.confidence:.2f}" if hasattr(det, "confidence") and det.confidence else ""
                eval_tag = det["eval"] if det.has_field("eval") else None
                eval_str = f" [{eval_tag}]" if eval_tag else ""
                ax.text(x1, y1 - 3, f"{det.label}{conf_str}{eval_str}",
                        color="white", fontsize=7, fontweight="bold",
                        bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.8))
        n_det = len(detections.detections) if detections and detections.detections else 0
        ax.set_title(f"{title}\n({n_det} detections)", fontsize=10)
        ax.axis("off")

    tp = sample["eval_tp"] if sample.has_field("eval_tp") else 0
    fp = sample["eval_fp"] if sample.has_field("eval_fp") else 0
    fn = sample["eval_fn"] if sample.has_field("eval_fn") else 0
    fig.suptitle(
        f"{Path(sample.filepath).name}   |   TP={tp}  FP={fp}  FN={fn}",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()

print("plot_gt_vs_pred() helper loaded.")

In [ ]:
# ── Cell 0c: Load YOLO dataset into FiftyOne ───────────────────────────────
import fiftyone as fo
from fiftyone import ViewField as F

print(f"FiftyOne version: {fo.__version__}")

# Safe re-run: delete existing dataset if it exists
if DATASET_NAME in fo.list_datasets():
    fo.delete_dataset(DATASET_NAME)

# FiftyOne's YOLOv5 importer needs the yaml path
yaml_path = DATASET_DIR / "dataset.yaml"
if not yaml_path.exists():
    yaml_path = DATASET_DIR / "data.yaml"
assert yaml_path.exists(), f"YAML not found at {DATASET_DIR} -- check your dataset!"

# Load the validation split with teacher labels as ground truth
dataset = fo.Dataset.from_dir(
    dataset_dir=str(DATASET_DIR),
    dataset_type=fo.types.YOLOv5Dataset,
    split="val",
    label_field="ground_truth",
    name=DATASET_NAME,
    yaml_path=str(yaml_path),
)

print(f"\nDataset: {dataset.name}")
print(f"Samples: {len(dataset)}")
print(f"Media type: {dataset.media_type}")

# Count ground truth labels by class
class_counts = dataset.count_values("ground_truth.detections.label")
print(f"\nGround truth labels (from teacher):")
for cls_name, count in sorted(class_counts.items()):
    print(f"  {cls_name:15s}  {count:4d} instances")
print(f"  {'TOTAL':15s}  {sum(class_counts.values()):4d} instances")

---
## Section 1: Run Student Model & Evaluate

We load a trained baseline model (or use pre-baked weights) and run inference on every
validation image. FiftyOne stores predictions alongside ground truth -- same images,
two sets of bounding boxes -- so we can compare them per-sample.

After evaluation, every sample gets three new fields:
- **eval_tp**: True Positives (student agrees with teacher)
- **eval_fp**: False Positives (student detects something the teacher did not label)
- **eval_fn**: False Negatives (teacher labeled something the student missed)

In [ ]:
# ── Cell 1a: Add student model predictions ────────────────────────────────
from ultralytics import YOLO

# Load trained student model
# If you have trained your own model, update BEST_WEIGHTS in cell 0a.
# Otherwise, use pre-baked weights from data/ppe_results/
assert BEST_WEIGHTS is not None and Path(BEST_WEIGHTS).exists(), (
    f"No trained weights found at {BEST_WEIGHTS}. "
    f"Either train a model first (Section 8) or check data/ppe_results/ for pre-baked weights."
)

student = YOLO(str(BEST_WEIGHTS))
print(f"Loaded student model from: {BEST_WEIGHTS}")
print(f"Model classes: {student.names}")

# Run inference on every sample and store predictions
dataset.apply_model(student, label_field="predictions")

# Count predictions
pred_counts = dataset.count_values("predictions.detections.label")
print(f"\nStudent predictions:")
for cls_name, count in sorted(pred_counts.items()):
    print(f"  {cls_name:15s}  {count:4d} instances")

gt_total = sum(class_counts.values())
pred_total = sum(pred_counts.values())
print(f"\nGround truth total: {gt_total}")
print(f"Prediction total:   {pred_total}")
print(f"Difference:         {pred_total - gt_total:+d}")

In [ ]:
# ── Cell 1b: Run COCO-style evaluation ────────────────────────────────────

# Clear any previous evaluation
if "eval" in dataset.list_evaluations():
    dataset.delete_evaluation("eval")

results = dataset.evaluate_detections(
    "predictions",
    gt_field="ground_truth",
    eval_key="eval",
    method="coco",
    iou=0.50,
    compute_mAP=True,
)

print("=" * 60)
print(f"  mAP@0.5 = {results.mAP():.4f}")
print("=" * 60)

print("\nPer-class evaluation report:")
print("-" * 60)
classes = list(CLASS_NAMES.values())
results.print_report(classes=classes)

### Think about it

- Which class has worse **recall**? What does low recall mean in practice for a safety system?
- If `person` recall is 90% but `hardhat` recall is 60%, what happens to compliance decisions?
- Is mAP the right metric for a safety application, or should we care more about recall?

---
## Section 2: Error Distribution Analysis

A single mAP number hides critical information. We need to understand:
- Are errors spread evenly across images, or concentrated in a few?
- Which specific images are the worst?
- When the student and teacher disagree, who is right?

### 2.1 TP / FP / FN Histograms

> **Think about it**: Before running this cell, predict: will errors be uniform
> across images, or will a few images dominate? What would each pattern imply
> for your data improvement strategy?

In [ ]:
# ── Cell 2a: Per-image error histograms ───────────────────────────────────

tp_values = dataset.values("eval_tp")
fp_values = dataset.values("eval_fp")
fn_values = dataset.values("eval_fn")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, values, title, color in zip(
    axes,
    [tp_values, fp_values, fn_values],
    ["True Positives", "False Positives", "False Negatives"],
    ["#27ae60", "#e74c3c", "#f39c12"],
):
    max_val = max(values) if values else 0
    ax.hist(values, bins=range(0, max_val + 2), color=color, edgecolor="black", alpha=0.8)
    ax.set_xlabel("Count per image")
    ax.set_ylabel("Number of images")
    ax.set_title(f"{title}\n(mean={np.mean(values):.1f}, max={max_val})")
    ax.grid(True, alpha=0.3)

plt.suptitle("Per-Image Detection Results", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### Think about it

- Are errors spread evenly or concentrated in a few images?
- What does a **long FP tail** tell you? (Hint: the student detects objects the teacher never labeled.)
- What does a **long FN tail** tell you? (Hint: the teacher labeled objects the student cannot find.)

### 2.2 Worst-Case Identification

> **Think about it**: Look at the filenames of the worst images below.
> Do they share visual characteristics? (scene type, camera angle, crowd density?)

In [ ]:
# ── Cell 2b: Top-5 worst images by FP and FN ──────────────────────────────

print("Top 5 images with most FALSE POSITIVES:")
print("  (Student detects objects the teacher did NOT label)")
print("  Could be hallucinations OR teacher omissions -- check visually below.\n")
worst_fp = dataset.sort_by("eval_fp", reverse=True)
for i, sample in enumerate(worst_fp[:5]):
    n_gt = len(sample["ground_truth"].detections) if sample["ground_truth"] else 0
    n_pred = len(sample["predictions"].detections) if sample["predictions"] else 0
    print(f"  {i+1}. FP={sample['eval_fp']}, FN={sample['eval_fn']}, TP={sample['eval_tp']}  "
          f"(GT={n_gt}, Pred={n_pred})  |  {Path(sample.filepath).name}")

print(f"\nTop 5 images with most FALSE NEGATIVES:")
print("  (Teacher labeled objects the student MISSED)\n")
worst_fn = dataset.sort_by("eval_fn", reverse=True)
for i, sample in enumerate(worst_fn[:5]):
    n_gt = len(sample["ground_truth"].detections) if sample["ground_truth"] else 0
    n_pred = len(sample["predictions"].detections) if sample["predictions"] else 0
    print(f"  {i+1}. FN={sample['eval_fn']}, FP={sample['eval_fp']}, TP={sample['eval_tp']}  "
          f"(GT={n_gt}, Pred={n_pred})  |  {Path(sample.filepath).name}")

### 2.3 Visualize the Worst Failures

> **Think about it**: For each image below, decide:
> 1. Is the **student** wrong? (missed a real object or hallucinated one)
> 2. Is the **teacher** wrong? (mislabeled or missed an object in the auto-labels)
> 3. If many FPs turn out to be real objects the teacher missed, what does that
>    tell you about the quality of your auto-labeler?

In [ ]:
# ── Cell 2c: Visualize 4 worst images by total errors ────────────────────

error_view = dataset.sort_by(F("eval_fp") + F("eval_fn"), reverse=True)

print("Worst 4 images by total detection errors (FP + FN):\n")
for sample in error_view[:4]:
    plot_gt_vs_pred(sample)

---
## Section 3: Label Quality Analysis

Standard evaluation tells us where the **student** fails. But in a distillation pipeline,
the ground truth came from an auto-labeler (the **teacher**). When the student disagrees
with the teacher, who is right?

FiftyOne's **mistakenness** algorithm answers this question:
- It compares the student's confident predictions against the teacher's labels
- If the student confidently disagrees, the teacher is likely wrong
- High mistakenness = likely annotation error from the auto-labeler

This is the core data-centric insight: **use the student to audit the teacher.**

### 3.1 Compute Mistakenness

> **Think about it**: The mistakenness score uses the student to audit the teacher.
> If the student **confidently** disagrees with a label, who is more likely wrong --
> the student (trained on hundreds of examples) or the teacher (ran once per image)?

In [ ]:
# ── Cell 3a: Compute mistakenness scores ─────────────────────────────────
import fiftyone.brain as fob

print("Computing mistakenness (likelihood that teacher labels are wrong)...")
print("This compares student predictions vs teacher labels per-sample.\n")

fob.compute_mistakenness(
    dataset,
    "predictions",
    label_field="ground_truth",
    mistakenness_field="mistakenness",
)

# Plot distribution
mistake_vals = dataset.values("mistakenness")
p90 = np.percentile(mistake_vals, 90)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(mistake_vals, bins=30, color="#e74c3c", edgecolor="black", alpha=0.8)
ax.set_xlabel("Mistakenness Score")
ax.set_ylabel("Number of Images")
ax.set_title("Label Mistakenness Distribution\n(High = likely annotation error from teacher)")
ax.axvline(x=p90, color="#2c3e50", linestyle="--", linewidth=2,
           label=f"90th percentile = {p90:.3f}")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Top 10 most suspicious samples
print("\nTop 10 most SUSPICIOUS samples (highest mistakenness):")
print("These are where the student CONFIDENTLY DISAGREES with the teacher.\n")
suspicious = dataset.sort_by("mistakenness", reverse=True)
for i, sample in enumerate(suspicious[:10]):
    tp = sample["eval_tp"] if sample.has_field("eval_tp") else 0
    fp = sample["eval_fp"] if sample.has_field("eval_fp") else 0
    fn = sample["eval_fn"] if sample.has_field("eval_fn") else 0
    print(f"  {i+1:2d}. mistakenness={sample['mistakenness']:.3f}  |  "
          f"TP={tp} FP={fp} FN={fn}  |  {Path(sample.filepath).name}")

### 3.2 Visualize Suspicious Samples

> **Think about it**: For each suspicious image below, make a judgment call:
> 1. **Fix the teacher's label** -- the auto-labeler got it wrong
> 2. **Student is wrong** -- the auto-label was correct, model is confused
> 3. **Ambiguous** -- hard to tell even for a human
>
> Keep a mental tally of your decisions. If most are (1), your auto-labeler needs work.

In [ ]:
# ── Cell 3b: Visualize top-3 suspicious samples ──────────────────────────

print("Top 3 most suspicious samples -- are the teacher's labels correct?\n")
suspicious = dataset.sort_by("mistakenness", reverse=True)
for sample in suspicious[:3]:
    plot_gt_vs_pred(sample)

---
## Section 4: Data Diversity Analysis

Are there redundant images in our dataset? Images that are near-duplicates add training
cost without adding information. FiftyOne's **uniqueness** score identifies them.

We also compute 2D embeddings to see the dataset's visual structure -- clusters,
outliers, and gaps.

### 4.1 Uniqueness Scores

> **Think about it**: If 20% of your dataset is near-duplicate, would removing them
> hurt or help? Form a hypothesis before looking at the results.

In [ ]:
# ── Cell 4a: Compute uniqueness scores ───────────────────────────────────

print("Computing uniqueness scores (visual distinctiveness of each image)...")
print("This computes image embeddings and compares all pairs -- may take 30-60 seconds.\n")

fob.compute_uniqueness(dataset, uniqueness_field="uniqueness")

# Plot distribution
uniqueness_vals = dataset.values("uniqueness")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(uniqueness_vals, bins=30, color="#3498db", edgecolor="black", alpha=0.8)
ax.set_xlabel("Uniqueness Score")
ax.set_ylabel("Number of Images")
ax.set_title("Image Uniqueness Distribution\n(Low = redundant, High = distinct)")
ax.axvline(x=np.median(uniqueness_vals), color="#e74c3c", linestyle="--",
           label=f"Median = {np.median(uniqueness_vals):.3f}")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 5 most redundant
print("\n5 most REDUNDANT images (candidates for removal):")
redundant = dataset.sort_by("uniqueness")
for i, sample in enumerate(redundant[:5]):
    print(f"  {i+1}. uniqueness={sample['uniqueness']:.3f}  |  {Path(sample.filepath).name}")

# 5 most unique
print("\n5 most UNIQUE images (highest information value):")
unique = dataset.sort_by("uniqueness", reverse=True)
for i, sample in enumerate(unique[:5]):
    print(f"  {i+1}. uniqueness={sample['uniqueness']:.3f}  |  {Path(sample.filepath).name}")

### 4.2 2D Embeddings Visualization

Embeddings project each image into a vector space, then reduce to 2D. This reveals:
- **Clusters**: groups of visually similar images (same scene type)
- **Outliers**: images very different from everything else
- **Distribution gaps**: visual conditions with no training data

> **Think about it**: Are errors spatially clustered? Do redundant images overlap
> with error-prone regions? If so, what does that tell you about what data to fix?

In [ ]:
# ── Cell 4b: Compute 2D embeddings ────────────────────────────────────────

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("umap-learn not installed. Falling back to PCA.")
    print("Install with: pip install umap-learn\n")

viz_method = "umap" if HAS_UMAP else "pca"
print(f"Computing 2D embeddings with {viz_method.upper()}...")
print("This extracts features from every image -- may take 1-2 minutes.\n")

viz_results = fob.compute_visualization(
    dataset,
    brain_key="img_viz",
    method=viz_method,
    num_dims=2,
    seed=51,
)

print(f"Visualization computed. {len(dataset)} points in 2D.")

In [ ]:
# ── Cell 4c: Three scatter plots colored by different metrics ───────────────

coords = np.array(viz_results.current_points)
assert coords.shape == (len(dataset), 2), f"Expected ({len(dataset)}, 2), got {coords.shape}"

uniqueness = np.array(dataset.values("uniqueness"))
mistakenness = np.array(dataset.values("mistakenness"))
total_errors = np.array(dataset.values("eval_fp")) + np.array(dataset.values("eval_fn"))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Colored by uniqueness
sc1 = axes[0].scatter(coords[:, 0], coords[:, 1], c=uniqueness,
                       cmap="RdYlGn", s=60, edgecolors="black", linewidths=0.3, alpha=0.8)
axes[0].set_title("Colored by Uniqueness\n(green=unique, red=redundant)")
plt.colorbar(sc1, ax=axes[0], label="Uniqueness")

# Plot 2: Colored by mistakenness
sc2 = axes[1].scatter(coords[:, 0], coords[:, 1], c=mistakenness,
                       cmap="RdYlGn_r", s=60, edgecolors="black", linewidths=0.3, alpha=0.8)
axes[1].set_title("Colored by Mistakenness\n(red=likely label error)")
plt.colorbar(sc2, ax=axes[1], label="Mistakenness")

# Plot 3: Colored by total errors
sc3 = axes[2].scatter(coords[:, 0], coords[:, 1], c=total_errors,
                       cmap="YlOrRd", s=60, edgecolors="black", linewidths=0.3, alpha=0.8)
axes[2].set_title("Colored by Total Errors (FP+FN)\n(dark=more errors)")
plt.colorbar(sc3, ax=axes[2], label="FP + FN")

for ax in axes:
    ax.set_xlabel(f"{viz_method.upper()}-1")
    ax.set_ylabel(f"{viz_method.upper()}-2")
    ax.grid(True, alpha=0.2)

plt.suptitle("Dataset Embeddings -- Three Perspectives", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Look for spatial patterns:")
print("  - Are errors clustered in one region? (= specific visual conditions cause failures)")
print("  - Are redundant images clustered? (= similar images from the same scene type)")
print("  - Are label errors near the boundary between clusters?")

---
## Section 5: Data Quality Dashboard

Let us combine all the metrics into a single dashboard view. This shows the
relationship between label quality (mistakenness), detection errors (FP+FN),
and data redundancy (uniqueness).

> **Think about it**: If you could only fix **5 images** before re-training,
> which 5 would you choose and why? (Top-right corner of the first plot = highest priority.)

In [ ]:
# ── Cell 5a: Combined quality scatter plots ───────────────────────────────

errors = np.array(dataset.values("eval_fp")) + np.array(dataset.values("eval_fn"))
mistake = np.array(dataset.values("mistakenness"))
uniq = np.array(dataset.values("uniqueness"))
names = [Path(fp).name for fp in dataset.values("filepath")]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Mistakenness vs Total Errors
ax = axes[0]
sc = ax.scatter(mistake, errors, c=uniq, cmap="RdYlGn", s=80,
                edgecolors="black", linewidths=0.5, alpha=0.8)
plt.colorbar(sc, ax=ax, label="Uniqueness")
ax.set_xlabel("Mistakenness", fontsize=11)
ax.set_ylabel("Total Errors (FP + FN)", fontsize=11)
ax.set_title("Label Quality Map\n(top-right = worst samples)")

# Annotate the worst points
worst_idx = np.argsort(mistake + errors / max(errors.max(), 1))[-5:]
for idx in worst_idx:
    ax.annotate(names[idx], (mistake[idx], errors[idx]),
                fontsize=7, alpha=0.8,
                textcoords="offset points", xytext=(5, 5))
ax.grid(True, alpha=0.3)

# Plot 2: Uniqueness vs Mistakenness
ax2 = axes[1]
sc2 = ax2.scatter(uniq, mistake, c=errors, cmap="YlOrRd", s=80,
                  edgecolors="black", linewidths=0.5, alpha=0.8)
plt.colorbar(sc2, ax=ax2, label="Total Errors")
ax2.set_xlabel("Uniqueness", fontsize=11)
ax2.set_ylabel("Mistakenness", fontsize=11)
ax2.set_title("Redundancy vs Label Quality\n(top-left = redundant AND wrong labels)")
ax2.grid(True, alpha=0.3)

plt.suptitle("Data Quality Dashboard", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary statistics
n_total = len(dataset)
n_perfect = int(np.sum(errors == 0))
n_high_mistake = int(np.sum(mistake > np.percentile(mistake, 80)))
n_low_unique = int(np.sum(uniq < np.percentile(uniq, 20)))

print(f"\n{'='*60}")
print(f"  DATA QUALITY SUMMARY")
print(f"{'='*60}")
print(f"  Total validation images:          {n_total}")
print(f"  Perfect agreement (0 errors):     {n_perfect} ({n_perfect/n_total*100:.0f}%)")
print(f"  High mistakenness (top 20%):      {n_high_mistake} images -- review labels")
print(f"  Low uniqueness (bottom 20%):      {n_low_unique} images -- consider removing")
print(f"  mAP@0.5:                          {results.mAP():.4f}")
print(f"{'='*60}")

---
## Section 6: Per-Scene Category Analysis

Our synthetic PPE dataset has images from different scene categories (cctv, warehouse,
highway, etc.). The scene category is encoded in the filename: `cctv_cctv_01.jpg`
has category `cctv`.

Let us check if the model performs differently across scene types.

> **Think about it**: Which scene type do you expect to have the **worst** performance?
> (Think about object size, occlusion, camera distance, lighting.)

In [ ]:
# ── Cell 6a: Extract scene category from filenames ───────────────────────

# Parse scene category from filenames like "cctv_cctv_01.jpg" -> "cctv"
# or "mixed_compliance_mixed_compliance_05.jpg" -> "mixed_compliance"
# or "edge_cases_edge_cases_03.jpg" -> "edge_cases"
#
# The auto-labeler flattens category/filename into category_filename,
# so filenames follow the pattern: {category}_{original_name}
# where original_name is {category}_{number}.
# Result: {category}_{category}_{number}

import re

def extract_category(stem: str) -> str:
    """Extract scene category from a filename stem like 'cctv_cctv_01'.

    Strategy: remove the trailing _NN number, then check if the
    remaining string is a doubled category (cat_cat). If the prefix
    has the form X_X, return X.
    """
    # Remove trailing _NN (one or more digits)
    m = re.match(r"^(.+)_(\d+)$", stem)
    if not m:
        return "unknown"
    prefix = m.group(1)  # e.g. "cctv_cctv" or "mixed_compliance_mixed_compliance"

    # Try to split prefix into two equal halves separated by _
    # Check all possible split points
    parts = prefix.split("_")
    for i in range(1, len(parts)):
        left = "_".join(parts[:i])
        right = "_".join(parts[i:])
        if left == right:
            return left

    # Fallback: return the full prefix
    return prefix


for sample in dataset.iter_samples():
    stem = Path(sample.filepath).stem
    category = extract_category(stem)
    sample["scene_category"] = category
    sample.save()

# Show categories found
categories = dataset.distinct("scene_category")
print(f"Scene categories found: {categories}")
print(f"\nImages per category:")
for cat in sorted(categories):
    count = len(dataset.match(F("scene_category") == cat))
    print(f"  {cat:25s}  {count:3d} images")


In [ ]:
# ── Cell 6b: Per-category mAP and error summary ──────────────────────────

print(f"{'Category':25s} {'Images':>7s} {'mAP@0.5':>9s} {'Avg FP':>8s} {'Avg FN':>8s} {'Avg Mistake':>12s}")
print("-" * 75)

cat_stats = []
for cat in sorted(categories):
    cat_view = dataset.match(F("scene_category") == cat)
    n_imgs = len(cat_view)
    
    # Compute mAP for this category
    eval_key = f"eval_{cat.replace(' ', '_')}"
    if eval_key in dataset.list_evaluations():
        dataset.delete_evaluation(eval_key)
    try:
        cat_results = cat_view.evaluate_detections(
            "predictions",
            gt_field="ground_truth",
            eval_key=eval_key,
            method="coco",
            iou=0.50,
            compute_mAP=True,
        )
        cat_mAP = cat_results.mAP()
    except Exception:
        cat_mAP = float("nan")
    
    avg_fp = np.mean(cat_view.values("eval_fp"))
    avg_fn = np.mean(cat_view.values("eval_fn"))
    avg_mistake = np.mean(cat_view.values("mistakenness"))
    
    print(f"  {cat:23s} {n_imgs:7d} {cat_mAP:9.3f} {avg_fp:8.1f} {avg_fn:8.1f} {avg_mistake:12.3f}")
    cat_stats.append({"category": cat, "n_images": n_imgs, "mAP": cat_mAP,
                      "avg_fp": avg_fp, "avg_fn": avg_fn, "avg_mistakenness": avg_mistake})

# Bar chart of per-category mAP
cat_df = pd.DataFrame(cat_stats).sort_values("mAP")

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#e74c3c" if m < 0.5 else "#f39c12" if m < 0.7 else "#27ae60" for m in cat_df["mAP"]]
ax.barh(cat_df["category"], cat_df["mAP"], color=colors, edgecolor="black", alpha=0.8)
ax.set_xlabel("mAP@0.5")
ax.set_title("Per-Scene Category Performance")
ax.axvline(x=results.mAP(), color="black", linestyle="--", alpha=0.5,
           label=f"Overall mAP={results.mAP():.3f}")
ax.legend()
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

### Think about it

- Which scene type has the **worst** performance? Why?
- Think about: object size, occlusion, camera distance, number of workers.
- Are the high-mistakenness scenes the same as the low-mAP scenes? What does that mean?

---
## Section 7: Data Curation Actions

Now that we understand our data quality issues, let us take action.

**Tiny label filtering** removes labels where either dimension is below a threshold.
These micro-boxes (often <20px at 640 resolution) are typically:
- False positives from the auto-labeler on background objects
- Truncated objects at image edges
- Objects too small for the model to learn from

In our reference experiments, filtering at 0.03 removed ~35% of labels and improved mAP50 by ~3%.
Your results may vary depending on which auto-labeler and prompts you used.


In [ ]:
# ── Cell 7a: Filter tiny labels ───────────────────────────────────────────
import subprocess

FILTERED_DIR = Path(str(DATASET_DIR) + "_filtered")
MIN_DIM = 0.03  # Try 0.02, 0.03, or 0.05 and compare label counts

print(f"Filtering tiny labels (min_dim={MIN_DIM})...")
print(f"  Input:  {DATASET_DIR}")
print(f"  Output: {FILTERED_DIR}")

result = subprocess.run(
    [
        sys.executable, str(DATA_DIR / "filter_tiny_labels.py"),
        "--input-dir", str(DATASET_DIR),
        "--output-dir", str(FILTERED_DIR),
        "--min-dim", str(MIN_DIM),
    ],
    capture_output=True, text=True,
    check=True,
)
print(result.stdout)


### Experiment: Try different thresholds

Change `MIN_DIM` above and re-run to see how different thresholds affect the label counts.
Try: `0.02`, `0.03`, `0.05`

- A lower threshold keeps more small objects (more data, more noise)
- A higher threshold is more aggressive (less noise, but may remove real small objects)

In [ ]:
# ── Cell 7b: Compare before/after filtering ───────────────────────────────

CLASS_COLORS_PLOT = {
    0: (0, 200, 0),      # hardhat -- green
    1: (220, 30, 30),     # person -- red
    2: (30, 100, 220),    # blue
}

def compare_before_after(image_path, label_before, label_after, class_names=None, figsize=(18, 7)):
    """Show an image with labels before and after filtering, side by side."""
    img = np.array(Image.open(image_path))
    h, w = img.shape[:2]

    def draw_labels(ax, label_path, title):
        ax.imshow(img)
        count = 0
        if Path(label_path).exists():
            for line in Path(label_path).read_text().strip().splitlines():
                parts = line.split()
                cls_id = int(parts[0])
                cx, cy, bw, bh = [float(x) for x in parts[1:5]]
                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w)
                y2 = int((cy + bh/2) * h)
                color = np.array(CLASS_COLORS_PLOT.get(cls_id, (200, 200, 200))) / 255
                name = class_names.get(cls_id, f"cls_{cls_id}") if class_names else f"cls_{cls_id}"
                rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                     linewidth=2, edgecolor=color, facecolor="none")
                ax.add_patch(rect)
                ax.text(x1, y1-4, name, color=color, fontsize=7, fontweight="bold",
                        bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.7))
                count += 1
        ax.set_title(f"{title} ({count} labels)", fontsize=11)
        ax.axis("off")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    draw_labels(ax1, label_before, "Before Filtering")
    draw_labels(ax2, label_after, "After Filtering")
    fig.suptitle(Path(image_path).name, fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Compare on a few validation images
if FILTERED_DIR.exists() and DATASET_DIR.exists():
    val_images = sorted((DATASET_DIR / "images" / "val").glob("*.*"))
    print(f"Comparing before/after on {min(3, len(val_images))} validation images:\n")
    for img_path in val_images[:3]:
        stem = img_path.stem
        compare_before_after(
            img_path,
            DATASET_DIR / "labels" / "val" / f"{stem}.txt",
            FILTERED_DIR / "labels" / "val" / f"{stem}.txt",
            class_names=CLASS_NAMES,
        )
else:
    print("Run the filtering step first (cell above).")

### Copilot Hook

Ask your AI copilot:

> "Based on the FiftyOne analysis above, what other data curation steps could improve
> my labels? I found high mistakenness in X images and low uniqueness in Y images.
> The worst scene category is Z. What would you recommend?"

### Phase 3 Observations

Fill in your findings:
- Labels removed by filtering: ___%
- Most common label issue: ___
- Images with highest mistakenness: ___
- Worst scene category: ___ (mAP = ___)
- If you could only fix 5 images, which 5 and why? ___

---
## Section 8: Training

Now we train a fast YOLO26n student model on the curated (filtered) labels.

| Parameter | Value | Why |
|-----------|-------|-----|
| Model | `yolo26n.pt` | Nano variant -- fast inference, small footprint |
| Epochs | 100 | With patience=20 early stopping |
| Batch | 8 | Safe for A40 GPU memory |
| Image size | 640 | Standard YOLO training resolution |
| Workers | 0 | Avoids multiprocessing issues on HPC |

**Time estimate**: ~5-15 minutes on A40.

**Short on time?** Pre-baked weights are available at `data/ppe_results/`.

In [ ]:
# ── Cell 8a: Train YOLO26n on curated labels ──────────────────────────────
# ============================================================
# TRAINING CONFIGURATION
# ============================================================
TRAIN_DATASET = FILTERED_DIR if FILTERED_DIR.exists() else DATASET_DIR
TRAIN_YAML = TRAIN_DATASET / "dataset.yaml"
EPOCHS = 100
PATIENCE = 20
BATCH = 8
IMGSZ = 640
TRAIN_DEVICE = DEVICE  # "cuda", "mps", or "cpu"
EXPERIMENT_NAME = "nb02_curated"
# ============================================================

assert TRAIN_YAML.exists(), f"Dataset YAML not found at {TRAIN_YAML}"
print(f"Training dataset: {TRAIN_DATASET}")
print(f"Config: epochs={EPOCHS}, patience={PATIENCE}, batch={BATCH}, imgsz={IMGSZ}, device={TRAIN_DEVICE}")

from ultralytics import YOLO

train_model = YOLO("yolo26n.pt")
train_results = train_model.train(
    data=str(TRAIN_YAML),
    epochs=EPOCHS,
    patience=PATIENCE,
    batch=BATCH,
    imgsz=IMGSZ,
    device=TRAIN_DEVICE,
    workers=0,
    project=str(DATA_DIR / "ppe_results"),
    name=EXPERIMENT_NAME,
)

print(f"\nTraining complete!")
print(f"Best weights: {train_results.save_dir}/weights/best.pt")


### While training runs...

Look back at your **Data Quality Dashboard** (Section 5). Based on what you found, would you:
- (a) Collect more data from the worst scene categories?
- (b) Fix specific labels flagged by mistakenness?
- (c) Try different auto-labeling prompts?
- (d) Change the filter threshold?

There is no single right answer. The point is to have a **hypothesis** about what
matters most, and then test it.

### Copilot Hook

Ask your AI copilot:

> "My training is running. While I wait, help me think through: my mAP was X,
> the worst scene category was Y with mAP Z, and Z% of labels had high mistakenness.
> What is the single highest-leverage change I could make for the next iteration?"

In [ ]:
# ── Cell 8b: Plot training curves ─────────────────────────────────────────

def plot_training_curves(results_csv, figsize=(16, 5)):
    """Plot training curves from Ultralytics results.csv."""
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    # Training loss
    loss_cols = [c for c in df.columns if "loss" in c.lower() and "val" not in c.lower()]
    ax = axes[0]
    for col in loss_cols:
        ax.plot(df["epoch"], df[col], label=col, alpha=0.8)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    # Validation loss
    val_loss_cols = [c for c in df.columns if "loss" in c.lower() and "val" in c.lower()]
    ax = axes[1]
    for col in val_loss_cols:
        ax.plot(df["epoch"], df[col], label=col, alpha=0.8)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Validation Loss")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    # mAP
    ax = axes[2]
    map_cols = [c for c in df.columns if "map" in c.lower() or "mAP" in c]
    for col in map_cols:
        ax.plot(df["epoch"], df[col], label=col, alpha=0.8, linewidth=2)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("mAP")
    ax.set_title("Validation mAP")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Best metrics
    if map_cols:
        best_col = [c for c in map_cols if "50" in c and "95" not in c]
        if best_col:
            best_epoch = df[best_col[0]].idxmax()
            best_val = df[best_col[0]].max()
            print(f"\nBest mAP50: {best_val:.4f} at epoch {int(df['epoch'].iloc[best_epoch])}")


# Find results.csv -- from our training or pre-baked
results_dir = DATA_DIR / "ppe_results" / EXPERIMENT_NAME
results_csv = results_dir / "results.csv"

if results_csv.exists():
    plot_training_curves(results_csv)
else:
    # Fallback to any pre-baked results
    prebaked = list((DATA_DIR / "ppe_results").glob("*/results.csv"))
    if prebaked:
        print(f"Using pre-baked results: {prebaked[-1].parent.name}")
        plot_training_curves(prebaked[-1])
    else:
        print("No results.csv found. Train a model first or check data/ppe_results/")

---
## Section 9: Post-Training Evaluation Loop

Now let us evaluate the freshly trained model using FiftyOne, and compare with
the baseline results from the beginning of this notebook.

In [ ]:
# ── Cell 9a: Evaluate the newly trained model ─────────────────────────────

# Find the best weights from the training run
new_weights = DATA_DIR / "ppe_results" / EXPERIMENT_NAME / "weights" / "best.pt"
if not new_weights.exists():
    # Fallback: use the original weights
    new_weights = BEST_WEIGHTS
    print(f"Using original weights: {new_weights}")
else:
    print(f"Using newly trained weights: {new_weights}")

# Load new model and add predictions
new_student = YOLO(str(new_weights))
dataset.apply_model(new_student, label_field="predictions_v2")

# Evaluate
if "eval_v2" in dataset.list_evaluations():
    dataset.delete_evaluation("eval_v2")

results_v2 = dataset.evaluate_detections(
    "predictions_v2",
    gt_field="ground_truth",
    eval_key="eval_v2",
    method="coco",
    iou=0.50,
    compute_mAP=True,
)

# Compare
mAP_baseline = results.mAP()
mAP_v2 = results_v2.mAP()
delta = mAP_v2 - mAP_baseline

print(f"\n{'='*60}")
print(f"  COMPARISON: Baseline vs Curated Training")
print(f"{'='*60}")
print(f"  Baseline mAP@0.5:         {mAP_baseline:.4f}")
print(f"  After curation mAP@0.5:   {mAP_v2:.4f}")
print(f"  Delta:                     {delta:+.4f} ({'+' if delta >= 0 else ''}{delta/max(mAP_baseline, 0.001)*100:.1f}%)")
print(f"{'='*60}")

print("\nPer-class report (after curation):")
print("-" * 60)
results_v2.print_report(classes=classes)

### Think about it

- Did your mAP improve? By how much?
- What was the biggest factor: the auto-labeler choice, the prompt, or the filtering?
- Which class improved more: `hardhat` or `person`? Why?
- If you had another iteration, what would you change?

### Copilot Hook

Ask your AI copilot:

> "My model went from mAP X to Y after data curation. The per-class results show...
> What would be the most impactful next step: (a) more data, (b) better auto-labeling
> prompts, (c) a larger model, or (d) different training hyperparameters? Explain why."

---
## Section 10: Phase 3-4 Reflection

### Observations

Fill in:
- Baseline mAP50: ___
- After curation mAP50: ___
- Training time: ___ minutes
- Best epoch: ___ (out of how many?)
- Did early stopping trigger? ___
- Worst scene category: ___
- Number of suspicious labels found: ___

### Key Takeaways

1. **Data-centric > model-centric.** We did not change the model architecture or hyperparameters.
   We improved the DATA, and the model improved.

2. **The student audits the teacher.** Mistakenness leverages the trained student to find
   where the auto-labeler made mistakes.

3. **Evaluation beyond mAP.** A single number hides critical information. Per-sample and
   per-scene analysis reveals exactly where to act.

4. **Embeddings reveal structure.** UMAP projections show whether errors cluster in
   specific visual conditions.

### What would you do differently in the next iteration?

___

---

**Next**: Open `03_evaluate_and_deploy.ipynb` to evaluate the model and build a compliance system.

In [ ]:
# ── Cleanup (optional) ───────────────────────────────────────────────────
# Uncomment to delete the FiftyOne dataset after analysis.
# fo.delete_dataset(DATASET_NAME)

print(f"Persistent FiftyOne datasets: {fo.list_datasets()}")
print(f"To clean up: fo.delete_dataset('{DATASET_NAME}')")